# 02 — Preprocessing: Normalisation & QC

Applies log-transform, low-expression / low-variance filters, and z-score normalisation.
Outputs a clean `processed_expression.csv` used by all downstream notebooks.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = '/content/gene-expression-la-analysis'
sys.path.insert(0, f'{PROJECT_ROOT}/src')

from preprocessing import (
    load_expression_csv, qc_report,
    filter_low_expression, filter_low_variance,
    remove_duplicate_genes, detect_outlier_samples,
    log2_transform, quantile_normalize, zscore_normalize,
    full_preprocessing_pipeline
)
from visualization import plot_expression_heatmap

sns.set_theme(style='whitegrid', font_scale=1.1)
GEO_ID = 'GSE2034'
print('Ready.')


In [ ]:
# ── Load raw matrix ────────────────────────────────────────────────────────
raw_path = f'{PROJECT_ROOT}/data/raw/{GEO_ID}_expression_raw.csv'
expr_raw = load_expression_csv(raw_path)
expr_raw = expr_raw.apply(pd.to_numeric, errors='coerce')

# Load metadata
meta = pd.read_csv(f'{PROJECT_ROOT}/data/raw/{GEO_ID}_metadata.csv', index_col=0)
print('Raw matrix:', expr_raw.shape)


## Step 1 – Log₂ Transform

In [ ]:
# Check if already log-scaled (mean < 20 suggests log already applied)
mean_val = expr_raw.mean().mean()
print(f'Global mean before transform: {mean_val:.2f}')

if mean_val > 20:
    expr_log = log2_transform(expr_raw, pseudocount=1.0)
    print('log2 transform applied')
else:
    expr_log = expr_raw.copy()
    print('Data appears already log-scaled; skipping transform')


In [ ]:
# ── Before / After distributions ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(expr_raw.values.flatten(), bins=80, color='grey', alpha=0.7)
axes[0].set_title('Before log2 Transform')
axes[0].set_xlabel('Expression')

axes[1].hist(expr_log.values.flatten(), bins=80, color='steelblue', alpha=0.7)
axes[1].set_title('After log2 Transform')
axes[1].set_xlabel('log2(Expression + 1)')

plt.suptitle('Expression Distribution Before / After log2', fontsize=12)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/02_log_transform.png',
            dpi=150, bbox_inches='tight')
plt.show()


## Step 2 – Outlier Sample Detection

In [ ]:
outliers = detect_outlier_samples(expr_log, z_threshold=3.0)

# Plot correlation-to-median for all samples
median_profile = expr_log.median(axis=1)
corrs = expr_log.apply(lambda col: col.corr(median_profile))

fig, ax = plt.subplots(figsize=(12, 3))
colors = ['crimson' if s in outliers else 'steelblue' for s in corrs.index]
ax.bar(range(len(corrs)), corrs.values, color=colors, alpha=0.8)
ax.axhline(corrs.mean() - 3*corrs.std(), ls='--', color='grey', label='±3σ')
ax.set_title('Sample Correlation to Median Profile')
ax.set_xlabel('Sample index')
ax.set_ylabel('Pearson r')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/02_outlier_samples.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Remove outliers
if outliers:
    expr_log = expr_log.drop(columns=outliers)
    meta     = meta.drop(index=[o for o in outliers if o in meta.index],
                         errors='ignore')
    print(f'Removed {len(outliers)} outlier sample(s). Remaining: {expr_log.shape[1]}')


## Step 3 – Gene Filtering

In [ ]:
expr_filt = filter_low_expression(expr_log, min_mean=1.0, min_expressed_frac=0.2)
expr_filt = filter_low_variance(expr_filt, variance_quantile=0.10)
expr_filt = remove_duplicate_genes(expr_filt)
print(f'After filtering: {expr_filt.shape}')


## Step 4 – Normalisation (Z-Score across genes)

In [ ]:
expr_norm = zscore_normalize(expr_filt, axis=0)   # per-gene across samples

# Check normalisation
print('Post-normalisation stats:')
print(f'  mean = {expr_norm.values.mean():.4f}  (should ≈ 0)')
print(f'  std  = {expr_norm.values.std():.4f}   (should ≈ 1)')


In [ ]:
# ── Post-normalisation boxplot ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
sample_data = [expr_norm.iloc[:, i].values for i in range(min(50, expr_norm.shape[1]))]
ax.boxplot(sample_data, patch_artist=True,
           boxprops=dict(facecolor='lightgreen', alpha=0.6),
           medianprops=dict(color='darkgreen', lw=1.5),
           flierprops=dict(marker='.', ms=2, alpha=0.3))
ax.set_title('Sample Distributions After Z-Score Normalisation (first 50 samples)')
ax.set_xlabel('Sample')
ax.set_ylabel('Z-score')
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/02_post_norm_boxplot.png',
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── QC report on processed matrix ─────────────────────────────────────────
_ = qc_report(expr_norm)


In [ ]:
# ── Save processed matrix ──────────────────────────────────────────────────
proc_path = f'{PROJECT_ROOT}/data/processed/processed_expression.csv'
meta_path = f'{PROJECT_ROOT}/data/processed/metadata_clean.csv'

expr_norm.to_csv(proc_path)
meta.to_csv(meta_path)

print(f'Saved processed matrix → {proc_path}')
print(f'Saved clean metadata   → {meta_path}')
print(f'Final matrix shape: {expr_norm.shape}')
